<a href="https://colab.research.google.com/github/eng-accelerator/ai-accelerator-C7/blob/main/Day%204/autoclasses.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# autoclasses - 2nd way to use hf models
# less abstract as compared to pipeline func that we covered last time

In [ ]:
! pip install datasets evaluate transformers diffusers accelerate ftfy pyarrow --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 3.8 MB/s eta 0:00:00


## Tokenizer

In [ ]:
# to create embeddings - right or wrong? wrong.
# smaller chucks for embeddings -
# need to use the corresponding embedding model - wrong.

In [ ]:
from transformers import AutoTokenizer # class

# tokenizers are specific to the models!
checkpoint = "distilbert-base-uncased-finetuned-sst-2-english"
# distilbert/distilbert-base-uncased-finetuned-sst-2-english
tokenizer = AutoTokenizer.from_pretrained(checkpoint) # i am loading the tokenizer ass w this model

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

In [ ]:
# send both of these inputs "together" to the model

raw_inputs = [
    "I am learning Operating Systems and it's so fun.", # more token ids
    "I write terrible C++ code!",  # lesser token ids
] # text inputs that the model cannot ingest directly

inputs = tokenizer(raw_inputs, padding=True, truncation=True, return_tensors="pt", max_length=15)
print(inputs)

{'input_ids': tensor([[ 101, 1045, 2572, 4083, 4082, 3001, 1998, 2009, 1005, 1055, 2061, 4569,
         1012,  102],
        [ 101, 1045, 4339, 6659, 1039, 1009, 1009, 3642,  999,  102,    0,    0,
            0,    0]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1],
        [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0]])}


In [ ]:
point 1: models work with matrices and vectors

point 2: 1st sent is longer than the 2nd sent -> 1st sent is going to have more token_ids as compared to 2nd

[4, 7, 8] # inp 1 -> row 1
[4, 9] # inp 2 -> row 2

[[4, 7, 8],
 [4, 9, 0]] # not a matrix

impossible for one to construct a matrix
something that helps us construct a matrix out of unequal inputs

In [ ]:
[4, 5, 6] # row 1 -> inp 1
[1, 2, <pad_tokens>] # row 2 -> inp 2




[4, 5, 6]
[1, 2, <missing>]


# unable to form a matrix -> matmul not possible -> model processing np

In [ ]:
# another way to do it - a closer look into the tokenizer
sequence = "I am learning Operating Systems and it's so fun."
tokens = tokenizer.tokenize(sequence)

print("--------TOKENIZED SENTENCE--------")
print(tokens)

print("\n\n--------TOKENS MAPPED TO THEIR IDS--------")
ids = tokenizer.convert_tokens_to_ids(tokens)
print(ids)

--------TOKENIZED SENTENCE--------
['i', 'am', 'learning', 'operating', 'systems', 'and', 'it', "'", 's', 'so', 'fun', '.']


--------TOKENS MAPPED TO THEIR IDS--------
[1045, 2572, 4083, 4082, 3001, 1998, 2009, 1005, 1055, 2061, 4569, 1012]


**TAKE A PAUSE!**

**SPECIAL TOKENS!**

In [ ]:
# already tokenized input

# mapping the token ids back to the tokens
decoded_string = tokenizer.decode([1045, 2572, 4083, 4082, 3001, 1998, 2009, 1005, 1055, 2061, 4569, 1012])
print(decoded_string)

i am learning operating systems and it's so fun.


In [ ]:
# raw text
decoded_string = tokenizer.decode([101, 1045, 2572, 4083, 4082, 3001, 1998, 2009, 1005, 1055, 2061, 4569,
         1012,  102])
print(decoded_string)

[CLS] i am learning operating systems and it's so fun. [SEP]


**Padding - ways to do it**

In [ ]:
# Will pad the sequences up to the maximum sequence length
model_inputs = tokenizer(sequences, padding="longest")

# 1st - 9 tokens, 11
# 2nd - 20 tokens, 0
# 3rd - 1 token, 19

# Will pad the sequences up to the model max length
# (512 for BERT or DistilBERT) model max length = the max sentence length that the model can ingest
model_inputs = tokenizer(sequences, padding="max_length")

# homework
# given max_length = 512
# 1st - 9 tokens,
# 2nd - 20 tokens,
# 3rd - 1 token,


# Will pad the sequences up to the specified max length
model_inputs = tokenizer(sequences, padding="max_length", max_length=8)

# homework
# given max_length = 8
# 1st - 9 tokens,
# 2nd - 20 tokens,
# 3rd - 1 token,




In [ ]:
Are these IDs generated by HF will be same as other LLMs?

## Models

In [ ]:
from transformers import AutoModel
import torch

checkpoint = "distilbert-base-uncased-finetuned-sst-2-english"
model = AutoModel.from_pretrained(checkpoint)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/629 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

In [ ]:
outputs = model(**inputs)
print(outputs.last_hidden_state.shape) # bs, seq_len, dim


# text gen decoder only models
# input w 5 tokens ids -> model (5 embeddings) -> output of the last layer (5 hidden states - 5 vectors) ----- (something)----- -> a logits vector of size V -> probs -> logprobs -> token prediction



torch.Size([2, 14, 768])


In [ ]:
from transformers import AutoModelForSequenceClassification
import torch

checkpoint = "distilbert-base-uncased-finetuned-sst-2-english" # this is a model for sentiment analysis
model = AutoModelForSequenceClassification.from_pretrained(checkpoint)
outputs = model(**inputs) # model doesn't expect a dictionary, it expects input_ids and attn_mask


print("--------SHAPE OF LOGIT TENSOR--------")
print(outputs.logits.shape)

print("\n\n--------LOGIT TENSOR--------")
print(outputs.logits)

# convert logits to probs --> softmax (math func)
predictions = torch.nn.functional.softmax(outputs.logits, dim=-1) # probs
predictions = torch.argmax(predictions, dim=1)
print("\n\n--------PREDICTIONS--------")
print(predictions) # os one, c++ one




--------SHAPE OF LOGIT TENSOR--------
torch.Size([2, 2])


--------LOGIT TENSOR--------
tensor([[-4.3357,  4.6875],
        [ 4.6717, -3.7884]], grad_fn=<AddmmBackward0>)


--------PREDICTIONS--------
tensor([1, 0])


In [ ]:
# homework: what is 1? negative or positive?
# figure out the mapping b/w numbers and their corresponding labels
# else post on circle to receive mentor help
# ans is in the api docs
# https://huggingface.co/distilbert/distilbert-base-uncased-finetuned-sst-2-english
